# 极坐标价量融合反转因子 — 逐段注释版 Notebook

说明：本 Notebook 基于 `exam_QR_周文熙.ipynb` 的代码结构，按段落逐步给出详细中文注释与解释。每一段先用 Markdown 解释该代码单元块的目的、输入输出、关键实现细节、常见错误与调试建议；随后给出对应的代码单元（与原代码逻辑一致，但在关键处增加注释以便阅读）。

使用建议：
- 如果你是初学者，先只读 Markdown 注释，理解整体流程；
- 如果要直接运行，请在与原数据 `exam_data.parquet` 同目录下打开，并执行代码单元。

注意：本文件以教学解释为主，保留原始函数与变量命名，便于对照原题目代码与 README 中的描述。

## 导入依赖库（目的与注意事项）

目的：引入本 Notebook 所需的 Python 库，并做一些 matplotlib / seaborn 的全局配置。

要点：
- 使用 matplotlib.use('Agg') 以保证在无显示环境（服务器/CI）下也能保存图形；在本地交互式环境可以移除或改为默认后端。
- 设置 plt.rcParams 保证图像分辨率与负号显示正确。
- 忽略警告可以让 Notebook 更整洁，但调试阶段建议打开以便发现潜在问题。

常见陷阱：
- 在交互式环境使用 Agg 后端可能无法直接弹窗显示图像，但文件仍会被保存。
- 若运行环境缺少某些库（如 seaborn），请先 pip install 对应包。

In [ ]:
from __future__ import annotations

import math
import warnings
from dataclasses import dataclass
from pathlib import Path

import matplotlib
matplotlib.use('Agg')  # 非交互式后端：在服务器/CI 环境避免 GUI 问题
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')  # 可选：便于阅读输出，但调试时建议注释掉
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 140
plt.rcParams['axes.unicode_minus'] = False

print('✓ 库导入完成（请确保已安装 pandas/numpy/matplotlib/seaborn）')

## 全局参数配置（Config）

目的：把所有可调实验参数集中管理，便于复现与快速调参。使用 dataclass 可以在类型检查与自动补全时更方便。

关键参数说明：
- data_path: 数据文件路径（exam_data.parquet）
- periods: 要计算的回看周期（20/60/120/240）
- cov_window: 协方差估计窗口（用于马氏距离中的协方差矩阵估计，题目建议 60）
- angle_target / angle_sigma: 角度权重函数的中心与宽度（控制 f(theta)）
- dynamic_lookback: 动态加权时用于平滑历史 RankIC 的回顾窗口

使用建议：在做敏感性分���时只需修改这里，然后重新运行 Notebook 的主体流程。

In [ ]:
@dataclass
class Config:
    data_path: str = 'exam_data.parquet'
    output_dir: str = 'outputs_notebook'
    periods: tuple[int, ...] = (20, 60, 120, 240)
    cov_window: int = 60          # 协方差估计窗口（天）
    fwd_horizon: int = 1          # 未来收益率持有期（天）
    quantiles: int = 5            # 分层回测的组数（例如 5 分位）
    angle_target: float = math.pi / 4  # 角度权重中心（45°）
    angle_sigma: float = math.pi / 4   # 角度权重的标准差（调节宽度）
    min_cross_section: int = 20   # 计算日度 IC 时最小截面样本量
    post_winsor_lower: float = 0.01
    post_winsor_upper: float = 0.99
    dynamic_lookback: int = 60    # 动态加权的回顾窗口（天）
    annual_trading_days: int = 252
    eps: float = 1e-12

cfg = Config()
print('配置: ', cfg)

## 工具函数（文件/打印/列名统一）

目的：这些函数并不直接影响因子计算逻辑，但对实验组织、输出保存和代码可读性非常重要。

说明：
- make_output_dirs：创建输出目录（tables/figures），确保保存文件时目录存在。
- print_section：在 Notebook 输出中清晰分隔不同阶段，便于读取。
- rename_columns：把数据集中常见的列名（trade_date, symbol）映射为代码中使用的统一列名（date, stock）。

In [ ]:
def make_output_dirs(root: Path) -> dict[str, Path]:
    paths = {
        'root': root,
        'tables': root / 'tables',
        'figures': root / 'figures',
    }
    for path in paths.values():
        path.mkdir(parents=True, exist_ok=True)
    return paths

def print_section(title: str) -> None:
    print('\n' + '=' * 96)
    print(title)
    print('=' * 96)

def rename_columns(df: pd.DataFrame) -> pd.DataFrame:
    # 把原始数据的常见列名映射为统一列名，便于后续代码一致使用
    rename_map = {
        'trade_date': 'date',
        'symbol': 'stock',
    }
    return df.rename(columns=rename_map)

## 1. 数据预处理（load_and_preprocess）—— 目的与要点

目的：
- 读取 parquet 数据，统一列名与类型，去除重复与关键缺失，过滤明显异常（如 close<=0、amount<=0），并计算常用辅助列（log_size, log_amount）。

实现要点：
- 日期转换要使用与数据一致的格式（题目给的是 YYYYMMDD 整数格式）；
- 在对数转换前先确保值 > 0，否则会产生 -inf / NaN；
- 数据排序（按 stock、date）是后续 groupby shift/rolling 的前提，否则会产生错位。

返回：清洗后的 DataFrame 与一份统计字典（用于快速检查数据质量）。

In [ ]:
def load_and_preprocess(cfg: Config) -> tuple[pd.DataFrame, dict]:
    data_path = Path(cfg.data_path)
    if not data_path.exists():
        raise FileNotFoundError(f'Cannot find data file: {data_path}')

    raw = pd.read_parquet(data_path)
    # 统一列名（如 trade_date -> date, symbol -> stock）
    raw = rename_columns(raw)

    # 检查必须的列
    required_cols = {'date', 'stock', 'close', 'amount'}
    missing = required_cols - set(raw.columns)
    if missing:
        raise ValueError(f'Missing required columns: {sorted(missing)}')

    before_rows = len(raw)
    duplicate_rows = raw.duplicated(subset=['date', 'stock']).sum()

    df = raw.copy()
    # 日期列转换：原为 YYYYMMDD（int），先转 str 再转 datetime
    df['date'] = pd.to_datetime(df['date'].astype(str), format='%Y%m%d', errors='coerce')
    df['stock'] = df['stock'].astype(str)

    # 把可能的数值列强制为数值型，errors='coerce' 会把非法值转为 NaN
    numeric_cols = [
        c
        for c in [
            'close', 'amount', 'volume', 'vwap', 'adj_factor',
            'open', 'high', 'low', 'pre_close', 'turnover', 'turnover_float',
            'mv', 'mv_float', 'new_ipo', 'limitupdown', 'limitupdown_at_close',
        ]
        if c in df.columns
    ]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # 统计关键字段的缺失比例（清洗前）
    missing_ratio = (
        df[[c for c in ['date', 'stock', 'close', 'amount'] if c in df.columns]]
        .isna()
        .mean()
    )

    # 去重并删除关键缺失行
    df = df.drop_duplicates(subset=['date', 'stock'], keep='last').copy()
    df = df.dropna(subset=['date', 'stock', 'close', 'amount']).copy()
    # 过滤掉明显异常（价格或成交额 <= 0）
    df = df[(df['close'] > 0) & (df['amount'] > 0)].copy()

    # 如果存在 volume 字段，可再过滤停牌（volume==0）的情况
    if 'volume' in df.columns:
        df = df[df['volume'].fillna(0) > 0].copy()

    # 如果存在次新股标志，可选择剔除
    if 'new_ipo' in df.columns:
        df = df[df['new_ipo'].fillna(0) == 0].copy()

    # 排序按 stock, date，这是后续 groupby(...).shift / rolling 的必要前提
    df = df.sort_values(['stock', 'date']).reset_index(drop=True)

    # 计算对数市值与对数成交额作为后续可能的中性化因子
    if 'mv_float' in df.columns:
        df['log_size'] = np.log(df['mv_float'].where(df['mv_float'] > 0))
    elif 'mv' in df.columns:
        df['log_size'] = np.log(df['mv'].where(df['mv'] > 0))
    else:
        df['log_size'] = np.nan
    df['log_amount'] = np.log(df['amount'])

    stats = {
        'rows_before': int(before_rows),
        'rows_after': int(len(df)),
        'duplicate_rows': int(duplicate_rows),
        'start_date': df['date'].min(),
        'end_date': df['date'].max(),
        'n_stocks': int(df['stock'].nunique()),
        'n_dates': int(df['date'].nunique()),
        'missing_ratio': missing_ratio,
        'describe': df[['close', 'amount']].describe().round(4),
    }
    return df, stats

## 构造未来收益（add_forward_return）

目的：为后续的因子检验（IC/RankIC/分层回测）构建标签，即 t 时点的因子是否能预测 t+H 的收益。

要点：
- 优先使用复权收盘价（adj_close = close * adj_factor）来计算未来收益，若数据中没有 adj_factor 则用未复权的 close。
- 使用 groupby('stock').shift(-horizon) 来取未来价格，必须保证数据已经按 stock/date 排序。
- 将 fwd_ret 写回 DataFrame，便于和因子值做横截面相关。

In [ ]:
def add_forward_return(df: pd.DataFrame, horizon: int = 1) -> pd.DataFrame:
    # 如果存在复权因子，则先计算复权收盘价
    if 'adj_factor' in df.columns:
        df['adj_close'] = df['close'] * df['adj_factor']
        future_adj_close = df.groupby('stock', sort=False)['adj_close'].shift(-horizon)
        df['fwd_ret'] = future_adj_close / df['adj_close'] - 1.0
        df['ret_source'] = 'adj_close'
    else:
        future_close = df.groupby('stock', sort=False)['close'].shift(-horizon)
        df['fwd_ret'] = future_close / df['close'] - 1.0
        df['ret_source'] = 'close'
    return df

## 2. 因子构建：协方差项准备（prepare_covariance_terms）

目的：为马氏距离计算准备协方差矩阵的元素（Var(close), Var(amount), Cov(close,amount)），使用过去 cfg.cov_window 天的滚动样本估计。

实现要点：
- 通过 grouped_rolling_mean 对每个股票分别计算滚动均值与二阶矩（E[x], E[x^2], E[xy]）。
- 用 Var(X) = E[X^2] - (E[X])^2 的公式得到方差，clip(lower=0) + eps 防止数值负值或 0 导致后续除零问题。
- cov_det = var_close * var_amount - cov^2 为协方差矩阵的行列式（判断矩阵是否可逆的标志）。若 cov_det 非正则置为 NaN，代表不可用。

注意：rolling 的 min_periods 设置为 window，意味着在数据不足的早期会有 NaN 覆盖，这是合理且可接受的。

In [ ]:
def grouped_rolling_mean(series: pd.Series, group: pd.Series, window: int) -> pd.Series:
    # 对 series 做按 group 的滚动均值，结果对齐回原索引
    return (
        series.groupby(group)
        .rolling(window=window, min_periods=window)
        .mean()
        .reset_index(level=0, drop=True)
    )

def prepare_covariance_terms(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    # group_key 为每行对应的股票标识，用于 rolling 计算
    group_key = df['stock']
    mean_close = grouped_rolling_mean(df['close'], group_key, cfg.cov_window)
    mean_amount = grouped_rolling_mean(df['amount'], group_key, cfg.cov_window)
    mean_close_sq = grouped_rolling_mean(df['close'] ** 2, group_key, cfg.cov_window)
    mean_amount_sq = grouped_rolling_mean(df['amount'] ** 2, group_key, cfg.cov_window)
    mean_cross = grouped_rolling_mean(
        df['close'] * df['amount'], group_key, cfg.cov_window
    )
    # 方差的无偏估计使用 E[x^2] - E[x]^2（在样本很小的情况下方差估计可能不准确）
    df['var_close'] = (mean_close_sq - mean_close**2).clip(lower=0) + cfg.eps
    df['var_amount'] = (mean_amount_sq - mean_amount**2).clip(lower=0) + cfg.eps
    df['cov_close_amount'] = mean_cross - mean_close * mean_amount
    # 协方差矩阵行列式，若接近 0 或负数表示矩阵不可逆或估计不稳定
    df['cov_det'] = df['var_close'] * df['var_amount'] - df['cov_close_amount'] ** 2
    df['cov_det'] = df['cov_det'].where(df['cov_det'] > cfg.eps, np.nan)
    return df

## 极角、角度权重与偏好系数 —— 理解与实现

目的：
- 通过 price_ret 与 amount_ret 计算极角 theta（用 arctan2），并以此为输入计算角度权重 f(theta) 与象限偏好系数 alpha。

要点：
- circular_distance：计算两个角在圆周上的最短距离（考虑 2π 环绕）。
- angle_weight：以高斯衰减形式把与目标角（45°）距离越近的角赋以越大权重。
- preference_alpha：根据 price_ret 与 amount_ret 的符号所在象限返回预设的 alpha 值（题目给定）。

调参说明：
- angle_sigma 控制 f(theta) 的宽度，sigma 越小意味着越集中在 45° 附近；sigma 越大意味着更宽容偏离。
- alpha 的值可以视为经济直觉的编码，例如把价跌量缩赋予较强的负权重（-1）等。

In [ ]:
def circular_distance(theta: pd.Series | np.ndarray, target: float) -> pd.Series | np.ndarray:
    diff = np.abs(theta - target)
    # 考虑圆周环绕，取最短的角距离
    return np.minimum(diff, 2 * np.pi - diff)

def angle_weight(theta: pd.Series, cfg: Config) -> pd.Series:
    dist = circular_distance(theta, cfg.angle_target)
    # 高斯衰减：以 target 为中心，sigma 控制宽度
    return np.exp(-(dist**2) / (2 * cfg.angle_sigma**2))

def preference_alpha(price_ret: pd.Series, amount_ret: pd.Series) -> pd.Series:
    # 根据 price_ret 与 amount_ret 的符号组合分配不同的 alpha
    alpha = np.select(
        [
            (price_ret >= 0) & (amount_ret >= 0),  # 第一象限：价升量增
            (price_ret < 0) & (amount_ret >= 0),   # 第二象限：价跌量增
            (price_ret < 0) & (amount_ret < 0),    # 第三象限：价跌量缩
            (price_ret >= 0) & (amount_ret < 0),   # 第四象限：价升量缩
        ],
        [1.0, -0.5, -1.0, 0.75],
        default=np.nan,
    )
    return pd.Series(alpha, index=price_ret.index)

## 单周期因子计算（build_single_period_factor）—— 逐步解释

目标：对给定回看周期 n（例如 20、60 等），计算单周期因子：
factor_n = alpha * rho * f(theta)

步骤详解：
1. 使用 groupby('stock').shift(n) 获取 n 日前的 close 与 amount（注意先排序）。
2. 计算绝对差 price_diff, amount_diff；以及相对变化 price_ret, amount_ret（用于 theta 与 alpha 的计算）。
3. theta = arctan2(amount_ret, price_ret) 并映射到 [0, 2π)。
4. alpha = preference_alpha(price_ret, amount_ret)。
5. f_theta = angle_weight(theta)。
6. 解析形式计算 rho^2：对 2x2 协方差矩阵 Σ = [[a, b], [b, c]]，有：
   rho^2 = (c * Δp^2 - 2 b Δp Δa + a * Δa^2) / (a c - b^2)
   代码直接使用这一公式避免逐行矩阵求逆。
7. 最终得到 factor_n，并将中间变量写回 df 以便分析。

数值稳定性说明：
- cov_det 在 prepare_covariance_terms 中已被设置为 NaN 如果不可用，rho 会在这些行上成为 NaN；
- 对 rho_sq < 0（数值误差）用 0 替代，避免出现复数；
- 把 inf/-inf 替换为 NaN。

In [ ]:
def build_single_period_factor(df: pd.DataFrame, n: int, cfg: Config) -> pd.DataFrame:
    # g 用于按股票做 shift
    g = df.groupby('stock', sort=False)
    # n 天前的价格与成交额
    close_lag = g['close'].shift(n)
    amount_lag = g['amount'].shift(n)

    # 绝对差与相对变化（相对变化用于角度计算）
    price_diff = df['close'] - close_lag
    amount_diff = df['amount'] - amount_lag
    price_ret = df['close'] / close_lag - 1.0
    amount_ret = df['amount'] / amount_lag - 1.0

    # 极角（映射到 [0, 2π)），以及 alpha / f(theta)
    theta = np.mod(np.arctan2(amount_ret, price_ret), 2 * np.pi)
    alpha = preference_alpha(price_ret, amount_ret)
    f_theta = angle_weight(theta, cfg)

    # 马氏距离平方的解析公式（避免逐点矩阵求逆）
    rho_sq = (
        df['var_amount'] * (price_diff**2)
        - 2 * df['cov_close_amount'] * price_diff * amount_diff
        + df['var_close'] * (amount_diff**2)
    ) / df['cov_det']
    # 数值保护：若 rho_sq 由于数值误差出现负值，则置为 0；若 cov_det 为 NaN 则 rho_sq 也为 NaN
    rho_sq = rho_sq.where(rho_sq >= 0, 0.0)
    rho = np.sqrt(rho_sq)

    factor_col = f'factor_{n}'
    # 把中间量写回 DataFrame，便于后续诊断
    df[f'price_ret_{n}'] = price_ret
    df[f'amount_ret_{n}'] = amount_ret
    df[f'theta_{n}'] = theta
    df[f'rho_{n}'] = rho
    df[factor_col] = alpha * rho * f_theta
    # 把 inf 替换为 NaN，防止异常值传播
    df[factor_col] = df[factor_col].replace([np.inf, -np.inf], np.nan)
    return df

def build_baseline_factors(df: pd.DataFrame, cfg: Config) -> tuple[pd.DataFrame, list[str]]:
    # 先准备协方差相关项
    df = prepare_covariance_terms(df, cfg)
    factor_cols: list[str] = []
    for n in cfg.periods:
        df = build_single_period_factor(df, n, cfg)
        factor_cols.append(f'factor_{n}')
    # 等权合成复合因子（baseline）
    df['factor_composite'] = df[factor_cols].mean(axis=1, skipna=True)
    return df, factor_cols

## 运行：数据清洗 + 因子构建（执行入口说明）

这里展示如何调用上述函数完成数据预处理与 baseline 因子构建。请确保数据文件存在且 cfg.data_path 指向正确路径。

输出：
- df：含有原始行情、future return、以及所有单周期与复合因子列的 DataFrame
- factor_cols：单周期因子列名列表（例如 factor_20, factor_60...）
- coverage：每个因子的覆盖率统计（用于诊断协方差或样本不足导致的缺失）

In [ ]:
print_section('1. 数据预处理')
df, stats = load_and_preprocess(cfg)
print(f"清洗前行数: {stats['rows_before']:,}")
print(f"清洗后行数: {stats['rows_after']:,}")
print(f"去重行数  : {stats['duplicate_rows']:,}")
print(f"日期范围  : {stats['start_date']} -> {stats['end_date']}")
print(f"股票数量  : {stats['n_stocks']:,}")
print(f"交易日数量: {stats['n_dates']:,}")
print('\n关键字段缺失比例（清洗前）:')
print(stats['missing_ratio'].round(4).to_string())
print('\n收盘价与成交额基本统计:')
print(stats['describe'].to_string())

df = add_forward_return(df, cfg.fwd_horizon)
print(f"\n未来收益率构造方式: {df['ret_source'].iloc[0]}")

print_section('2. 因子构建')
df, factor_cols = build_baseline_factors(df, cfg)
baseline_cols = factor_cols + ['factor_composite']
coverage = pd.DataFrame({
    'factor': baseline_cols,
    'coverage_ratio': [df[col].notna().mean() for col in baseline_cols]
})
print('因子覆盖率:')
print(coverage.round(4).to_string(index=False))

## 3. 因子检验：IC / RankIC / 分层回测（实现细节）

本节实现日度 IC（Pearson）与 RankIC（Spearman），并实现分层回测（按日把股票分为 Q 组并计算每组的平均未来收益）。

重要实现点：
- safe_corr：在计算相关前进行样本数与唯一值的检查，避免返回误导性相关。
- daily_ic_series：按日期做横截面相关，若当天样本 < cfg.min_cross_section 则返回 NaN（跳过当天）。
- assign_quantile_labels：先用 rank 避免重复值导致 qcut 无法分组的问题，再用 qcut 做等频分组。
- quantile_backtest：计算每日每组的平均未来收益，生成累计净值并计算 Qmax-Qmin 多空统计。

注意：在真实回测中需考虑交易成本、换手、容量等；本题只做无成本的理想化检验。

In [ ]:
def safe_corr(x: pd.Series, y: pd.Series, method: str) -> float:
    valid = pd.concat([x, y], axis=1).dropna()
    if len(valid) == 0:
        return np.nan
    # 如果某列只有一个唯一值，相关性没有意义
    if valid.iloc[:, 0].nunique() < 2 or valid.iloc[:, 1].nunique() < 2:
        return np.nan
    return valid.iloc[:, 0].corr(valid.iloc[:, 1], method=method)

def daily_ic_series(
    df: pd.DataFrame,
    factor_col: str,
    ret_col: str,
    method: str,
    min_cross_section: int,
) -> pd.Series:
    # 取出当天因子与未来收益的有效样本
    sub = df[['date', factor_col, ret_col]].dropna().copy()
    def _corr(grp: pd.DataFrame) -> float:
        if len(grp) < min_cross_section:
            return np.nan
        return safe_corr(grp[factor_col], grp[ret_col], method)
    result = sub.groupby('date', sort=True).apply(_corr)
    result.name = f'{factor_col}_{method}'
    return result.dropna()

def summarize_series(series: pd.Series, annual_days: int) -> dict:
    series = series.dropna()
    if series.empty:
        return {
            'mean': np.nan, 'std': np.nan, 't_stat': np.nan,
            'ir': np.nan, 'positive_ratio': np.nan, 'n_days': 0,
        }
    mean = series.mean()
    std = series.std(ddof=1)
    t_stat = mean / (std / np.sqrt(len(series)) + 1e-12)
    ir = mean / (std + 1e-12) * np.sqrt(annual_days)
    return {
        'mean': float(mean), 'std': float(std), 't_stat': float(t_stat),
        'ir': float(ir), 'positive_ratio': float((series > 0).mean()),
        'n_days': int(len(series)),
    }

def assign_quantile_labels(series: pd.Series, n_quantiles: int) -> pd.Series:
    # 返回与原 series 索引对齐的分位标签（1..n_quantiles），无法分组返回全 NaN
    result = pd.Series(np.nan, index=series.index)
    valid = series.dropna()
    if len(valid) < n_quantiles or valid.nunique() < n_quantiles:
        return result
    # 先 rank 然后 qcut，避免重复值导致 qcut 报错
    ranked = valid.rank(method='first')
    labels = pd.qcut(ranked, n_quantiles, labels=False) + 1
    result.loc[valid.index] = labels.astype(float)
    return result

def quantile_backtest(
    df: pd.DataFrame,
    factor_col: str,
    ret_col: str,
    n_quantiles: int,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, dict]:
    sub = df[['date', factor_col, ret_col]].copy()
    # 按日把因子值划分到分位
    sub['quantile'] = sub.groupby('date')[factor_col].transform(
        lambda s: assign_quantile_labels(s, n_quantiles)
    )
    # 计算每个日期、每组的平均未来收益
    group_ret = (
        sub.dropna(subset=['quantile', ret_col])
        .groupby(['date', 'quantile'])[ret_col]
        .mean()
        .unstack()
        .sort_index()
    )
    if group_ret.empty:
        empty_stats = {
            'ann_return': np.nan, 'ann_vol': np.nan, 'sharpe': np.nan,
            'max_drawdown': np.nan, 'win_rate': np.nan,
        }
        return group_ret, group_ret, pd.Series(dtype=float), empty_stats
    # 累计收益（假设日收益小且连续复利）
    cum_ret = (1 + group_ret.fillna(0)).cumprod()
    # 多空组合：最高分位 - 最低分位
    long_short = group_ret[n_quantiles] - group_ret[1]
    ls_stats = portfolio_stats(long_short)
    return group_ret, cum_ret, long_short, ls_stats

def portfolio_stats(ret: pd.Series, annual_days: int = 252) -> dict:
    # 计算策略的年化收益、年化波动、Sharpe、最大回撤、胜率等
    ret = ret.dropna()
    if ret.empty:
        return {
            'ann_return': np.nan, 'ann_vol': np.nan, 'sharpe': np.nan,
            'max_drawdown': np.nan, 'win_rate': np.nan,
        }
    nav = (1 + ret).cumprod()
    ann_return = nav.iloc[-1] ** (annual_days / len(nav)) - 1
    ann_vol = ret.std(ddof=0) * np.sqrt(annual_days)
    sharpe = ann_return / (ann_vol + 1e-12)
    drawdown = nav / nav.cummax() - 1
    max_drawdown = drawdown.min()
    win_rate = (ret > 0).mean()
    return {
        'ann_return': float(ann_return), 'ann_vol': float(ann_vol),
        'sharpe': float(sharpe), 'max_drawdown': float(max_drawdown),
        'win_rate': float(win_rate),
    }

## 绘图函数（plot_ic / plot_quantiles）—— 目的与注意事项

说明：绘制 IC 时序图以及分位累计收益与多空组合净值。函数会把图保存到指定路径。

注意：
- 若使用 Agg 后端，plt.show() 可能不会弹出窗口，但 plt.savefig 会正常保存文件；
- 保存后最好检查 outputs_notebook/figures 是否包含对应图像以保证后续展示。

In [ ]:
def plot_ic(ic: pd.Series, title: str, save_path: Path) -> None:
    if ic.empty:
        return
    rolling = ic.rolling(20, min_periods=5).mean()
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(ic.index, ic.values, label='Daily IC', alpha=0.35, linewidth=1.0)
    ax.plot(rolling.index, rolling.values, label='20D Rolling Mean', linewidth=2.0)
    ax.axhline(0, color='black', linestyle='--', linewidth=1.0)
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

def plot_quantiles(cum_ret: pd.DataFrame, long_short: pd.Series, title: str, save_path: Path) -> None:
    if cum_ret.empty:
        return
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for col in cum_ret.columns:
        axes[0].plot(cum_ret.index, cum_ret[col], label=f'Q{int(col)}', linewidth=1.5)
    axes[0].set_title(f'{title} Quantile Cumulative Return')
    axes[0].legend()
    ls_nav = (1 + long_short.fillna(0)).cumprod()
    axes[1].plot(ls_nav.index, ls_nav.values, color='#d62728', linewidth=2.0)
    axes[1].axhline(1.0, color='black', linestyle='--', linewidth=1.0)
    axes[1].set_title(f'{title} Long-Short')
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

## evaluate_factor：统一的因子评估流程（细节解释）

功能：给定一个因子列名，执行完整的评估流程并保存中间结果：
- 计算 daily IC 与 daily RankIC（并做统计汇总）
- 做分层回测并计算多空组合统计
- 保存 ic/rankic/csv、绘图文件

返回：一个字典行，包含所有关键统计结果，便于后续把多因子结果拼成表格进行比较。

使用建议：evaluate_factor 应在所有因子列都准备好之后统一调用，以便对比 baseline 与优化后因子。

In [ ]:
print_section('3. 因子检验（第一题 baseline）')
out_dirs = make_output_dirs(Path(cfg.output_dir))

def evaluate_factor(df: pd.DataFrame, factor_col: str, cfg: Config, out_dirs: dict[str, Path]) -> dict:
    # 计算 Pearson IC 与 Spearman RankIC 的日度序列
    ic = daily_ic_series(df, factor_col, 'fwd_ret', 'pearson', cfg.min_cross_section)
    rankic = daily_ic_series(df, factor_col, 'fwd_ret', 'spearman', cfg.min_cross_section)
    ic_summary = summarize_series(ic, cfg.annual_trading_days)
    rankic_summary = summarize_series(rankic, cfg.annual_trading_days)
    # 分层回测
    group_ret, cum_ret, long_short, ls_stats = quantile_backtest(df, factor_col, 'fwd_ret', cfg.quantiles)
    # 保存中间数据
    if not ic.empty:
        ic.to_csv(out_dirs['tables'] / f'{factor_col}_daily_ic.csv')
    if not rankic.empty:
        rankic.to_csv(out_dirs['tables'] / f'{factor_col}_daily_rankic.csv')
    if not group_ret.empty:
        group_ret.to_csv(out_dirs['tables'] / f'{factor_col}_quantile_return.csv')
    # 绘图
    plot_ic(ic, f'{factor_col} IC', out_dirs['figures'] / f'{factor_col}_ic.png')
    plot_quantiles(cum_ret, long_short, factor_col, out_dirs['figures'] / f'{factor_col}_quantile.png')
    return {
        'factor': factor_col,
        'ic_mean': ic_summary['mean'], 'ic_std': ic_summary['std'],
        'ic_ir': ic_summary['ir'], 'ic_tstat': ic_summary['t_stat'],
        'ic_pos_ratio': ic_summary['positive_ratio'],
        'rankic_mean': rankic_summary['mean'], 'rankic_std': rankic_summary['std'],
        'rankic_ir': rankic_summary['ir'], 'rankic_tstat': rankic_summary['t_stat'],
        'rankic_pos_ratio': rankic_summary['positive_ratio'],
        'ls_ann_return': ls_stats['ann_return'], 'ls_ann_vol': ls_stats['ann_vol'],
        'ls_sharpe': ls_stats['sharpe'], 'ls_max_drawdown': ls_stats['max_drawdown'],
        'ls_win_rate': ls_stats['win_rate'],
        'coverage': float(df[factor_col].notna().mean()),
    }

summary_rows = []
for col in baseline_cols:
    row = evaluate_factor(df, col, cfg, out_dirs)
    summary_rows.append(row)
    print(f"{col:20s} IC={row['ic_mean']:.6f} RankIC={row['rankic_mean']:.6f} LS AnnRet={row['ls_ann_return']:.6f}")

summary_df = pd.DataFrame(summary_rows).sort_values(['rankic_mean', 'ic_mean'], ascending=False)
summary_df.to_csv(out_dirs['tables'] / 'baseline_factor_evaluation.csv', index=False)
print('\n详细统计已保存至:', out_dirs['tables'] / 'baseline_factor_evaluation.csv')

## 4. 优化实验（第二题选作）—— 优化方向 A：后处理（winsorize / zscore / neutralize）

目的：验证原始因子是否被极端值或风格暴露（市值/行业）主导，若是，后处理可能提升稳定性与选股能力。

实现说明：
- winsorize_by_date：按日截面对因子值做上下分位裁剪（默认 1%/99%）。
- zscore_by_date：按日做 Z-score（x-mean)/std，保证不同日期的分布可比。
- neutralize_cross_section：按日做截面回归，剔除对数市值与行业哑变量的线性暴露，取残差作为中性化后的因子。

注意：中性化需要足够样本数（自变量个数不能超过样本数），否则该日跳过中性化处理。

In [ ]:
def winsorize_by_date(series: pd.Series, dates: pd.Series, lower: float, upper: float) -> pd.Series:
    def _clip(x: pd.Series) -> pd.Series:
        if x.dropna().empty:
            return x
        lo = x.quantile(lower)
        hi = x.quantile(upper)
        return x.clip(lo, hi)
    return series.groupby(dates).transform(_clip)

def zscore_by_date(series: pd.Series, dates: pd.Series) -> pd.Series:
    # ddof=0 与 numpy 的 std 一致，这里加上小量避免除零
    return series.groupby(dates).transform(lambda x: (x - x.mean()) / (x.std(ddof=0) + 1e-12))

def neutralize_cross_section(df: pd.DataFrame, factor_col: str, size_col: str = 'log_size', industry_col: str = 'ind_code') -> pd.Series:
    def _neutralize_one_day(grp: pd.DataFrame) -> pd.Series:
        # 返回与 grp.index 对齐的残差 series
        result = pd.Series(np.nan, index=grp.index)
        y = grp[factor_col]
        if y.notna().sum() < 10:
            # 样本不足时不做回归
            return result
        X_parts = [pd.DataFrame({'intercept': 1.0}, index=grp.index)]
        if size_col in grp.columns and grp[size_col].notna().sum() > 0:
            X_parts.append(pd.DataFrame({size_col: grp[size_col]}, index=grp.index))
        if industry_col in grp.columns and grp[industry_col].notna().sum() > 0:
            dummies = pd.get_dummies(grp[industry_col].astype(str), prefix='ind', drop_first=True)
            if not dummies.empty:
                X_parts.append(dummies)
        X = pd.concat(X_parts, axis=1)
        valid = y.notna() & X.notna().all(axis=1)
        if valid.sum() <= X.shape[1]:
            return result
        # 最小二乘求解 beta
        beta = np.linalg.lstsq(X.loc[valid].to_numpy(dtype=float), y.loc[valid].to_numpy(dtype=float), rcond=None)[0]
        fitted = X.loc[valid].to_numpy(dtype=float) @ beta
        result.loc[valid] = y.loc[valid].to_numpy(dtype=float) - fitted
        return result
    # 对每个交易日单独回归
    return df.groupby('date', group_keys=False).apply(_neutralize_one_day)

def optimize_postprocess(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    base = 'factor_composite'
    # 去极值 -> 标准化 -> 中性化 -> 再标准化
    df['factor_post_winsor'] = winsorize_by_date(df[base], df['date'], cfg.post_winsor_lower, cfg.post_winsor_upper)
    df['factor_post_z'] = zscore_by_date(df['factor_post_winsor'], df['date'])
    df['factor_post'] = neutralize_cross_section(df, 'factor_post_z')
    df['factor_post'] = zscore_by_date(df['factor_post'], df['date'])
    return df

print_section('4.1 后处理优化')
df = optimize_postprocess(df, cfg)
row_post = evaluate_factor(df, 'factor_post', cfg, out_dirs)
print(f"factor_post IC={row_post['ic_mean']:.6f} RankIC={row_post['rankic_mean']:.6f} LS AnnRet={row_post['ls_ann_return']:.6f}")

## 优化方向 B：动态加权复合（基于滚动 RankIC）—— 思路与实现细节

核心假设：不同回看周期因子的有效性会随时间变化，我们可以利用历史 RankIC（关注排序能力）来分配权重，从而让复合因子在不同市场环境下自适应。

实现要点：
- 对每个单周期因子计算 daily RankIC 序列，并拼成 panel（日期 × 因子列）。
- 做 rolling 平滑（rolling(cfg.dynamic_lookback).mean()），然后 shift(1) 防止前视偏差。
- 将负的历史 RankIC 截为 0（设计选择，表示不信任负指标），再按行归一化得到权重；若历史数据不足则回退到等权。
- 用每日报的权重去对已做截面 zscore 的单周期因子加权求和，得到 factor_dynamic，再做一次截面 zscore 保持可比性。

注意事项：
- rolling 的 min_periods 需要设置合理，避免初期全部 NaN；
- shift(1) 非常关键，若不 shift 会造成使用当天的信息去分配权重（前视）。

In [ ]:
def optimize_dynamic_weight(df: pd.DataFrame, factor_cols: list[str], cfg: Config) -> tuple[pd.DataFrame, pd.DataFrame]:
    # 计算每个单周期因子的历史 RankIC 序列
    rankic_map = {}
    for col in factor_cols:
        rankic_map[col] = daily_ic_series(df, col, 'fwd_ret', 'spearman', cfg.min_cross_section)
    rankic_panel = pd.concat(rankic_map, axis=1).sort_index()
    # 滚动平均平滑历史表现
    rolling_score = rankic_panel.rolling(cfg.dynamic_lookback, min_periods=max(20, cfg.dynamic_lookback // 3)).mean()
    # shift(1) 防止使用当天信息
    rolling_score = rolling_score.shift(1).clip(lower=0)
    # 行归一化得到权重
    weight_df = rolling_score.div(rolling_score.sum(axis=1), axis=0)
    if not weight_df.empty:
        # 对于 NaN 日（早期窗口不足），回退到等权
        equal_weight = 1.0 / len(factor_cols)
        weight_df = weight_df.fillna(equal_weight)
    # 对单周期因子先做截面 zscore（用于加权组合时尺度一致）
    z_cols = []
    for col in factor_cols:
        z_col = f'{col}_zopt'
        df[z_col] = zscore_by_date(df[col], df['date'])
        z_cols.append(z_col)
    weight_df = weight_df.reset_index().rename(columns={'index': 'date'})
    rename_weights = {col: f'weight_{col}' for col in factor_cols}
    weight_df = weight_df.rename(columns=rename_weights)
    # 把权重表 merge 回主表
    df = df.merge(weight_df, on='date', how='left')
    composite = 0
    total_weight = 0
    for col, z_col in zip(factor_cols, z_cols):
        w_col = rename_weights[col]
        composite = composite + df[z_col] * df[w_col]
        total_weight = total_weight + df[w_col]
    # 除以权重和（若为 0 则变为 NaN）
    df['factor_dynamic'] = composite / total_weight.replace(0, np.nan)
    df['factor_dynamic'] = zscore_by_date(df['factor_dynamic'], df['date'])
    return df, weight_df

print_section('4.2 动态加权优化')
df, weight_df = optimize_dynamic_weight(df, factor_cols, cfg)
weight_df.to_csv(out_dirs['tables'] / 'dynamic_weights.csv', index=False)
row_dynamic = evaluate_factor(df, 'factor_dynamic', cfg, out_dirs)
print(f"factor_dynamic IC={row_dynamic['ic_mean']:.6f} RankIC={row_dynamic['rankic_mean']:.6f} LS AnnRet={row_dynamic['ls_ann_return']:.6f}")
print('动态权重表已保存至:', out_dirs['tables'] / 'dynamic_weights.csv')

## 5. 全因子评价汇总与保存

目的：把 baseline 因子与优化后因子的评估结果汇总到一个表格中，便于横向比较并导出报告。

同时把最终的因子面板（date, stock, fwd_ret 与因子列）保存为 parquet，便于后续分析或复现。

In [ ]:
print_section('5. 全因子评价汇总')
eval_cols = baseline_cols + ['factor_post', 'factor_dynamic']
final_summary_rows = []
for col in eval_cols:
    row = evaluate_factor(df, col, cfg, out_dirs)
    final_summary_rows.append(row)

final_summary_df = pd.DataFrame(final_summary_rows).sort_values(['rankic_mean', 'ic_mean'], ascending=False)
final_summary_df.to_csv(out_dirs['tables'] / 'final_factor_evaluation.csv', index=False)

print(final_summary_df.round(6).to_string(index=False))
print('\n详细结果已保存至:', out_dirs['tables'] / 'final_factor_evaluation.csv')

# 保存因子面板（便于后续分析）
factor_panel_cols = ['date', 'stock', 'fwd_ret'] + eval_cols
df[factor_panel_cols].to_parquet(out_dirs['tables'] / 'factor_panel.parquet', index=False)
print('因子面板已保存至:', out_dirs['tables'] / 'factor_panel.parquet')

print_section('输出文件清单')
print('表格目录:', out_dirs['tables'].resolve())
print('图表目录:', out_dirs['figures'].resolve())

## 6. 关键图表预览（若已生成）

说明：本单元尝试读取并显示 outputs_notebook/figures 下的关键图像（复合因子的 IC 与分层图）。在无 GUI 或未生成图像时会提示用户。

In [ ]:
import matplotlib.image as mpimg
import os

fig_dir = out_dirs['figures']
ic_path = fig_dir / 'factor_composite_ic.png'
quantile_path = fig_dir / 'factor_composite_quantile.png'

if ic_path.exists() and quantile_path.exists():
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    img_ic = mpimg.imread(ic_path)
    img_qt = mpimg.imread(quantile_path)
    axes[0].imshow(img_ic)
    axes[0].axis('off')
    axes[0].set_title('Composite Factor IC')
    axes[1].imshow(img_qt)
    axes[1].axis('off')
    axes[1].set_title('Composite Factor Quantile')
    plt.tight_layout()
    plt.show()
else:
    print('图表文件未找到，请先运行上方代码生成图表。')

## 7. 结论与讨论（写作提示）

该部分为写作与口述建议（非代码）：
- 明确区分 baseline（题目要求）与后处理/优化（第二题选作）；
- 展示关键指标（IC mean, RankIC mean, IR, Q5-Q1 多空收益、Sharpe、回撤）并解释其含义；
- 若 baseline 表现不佳，优先检查覆盖率、样本量、极值与风格暴露是否影响结果，然后再尝试优化；
- 在面试中能清晰地陈述研究流程与数学公式（马氏距离解析式、角度权重函数等）非常关键。

如果你需要，我可以把本 Notebook 导出为 .ipynb 文件供下载，或把其中某个函数（例如 build_single_period_factor 或 neutralize_cross_section）做逐行代码注释并给出改进建议（例如用对数差替代绝对差、或者引入 shrinkage 协方差估计）。请告诉我你更想深入哪一部分。